In [9]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

train = pd.read_csv('../data/raw/train.csv')
test = pd.read_csv('../data/raw/test.csv')

X = train.drop(columns=['id', 'health_condition'])
y = train['health_condition']

In [10]:
def add_features(df):
    df = df.copy()
    df['low_stress_and_active'] = (
        (df['stress_level'] == 'low') & (df['physical_activity_level'] == 'active')
    ).astype(int)
    return df

X = add_features(X)

In [11]:
ordinal_cols = ['stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol']
ordinal_orders = [
    ['low', 'medium', 'high'],
    ['poor', 'average', 'good'],
    ['sedentary', 'moderate', 'active'],
    ['no', 'occasional', 'yes'],
]
onehot_cols = ['diet_type', 'gender']
numeric_cols = [c for c in X.columns if c not in ordinal_cols + onehot_cols]

preprocess = ColumnTransformer([
    ('num', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
    ]), numeric_cols),
    ('ord', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('encode', OrdinalEncoder(categories=ordinal_orders)),
    ]), ordinal_cols),
    ('cat', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('encode', OneHotEncoder(handle_unknown='ignore')),
    ]), onehot_cols),
])

In [12]:
model = Pipeline([
    ('prep', preprocess),
    ('clf', LogisticRegression(max_iter=1000)),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
print(scores.round(5))
print(f"mean: {scores.mean():.5f}  floor to beat: 0.86354")

/home/vscode/.local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/home/vscode/.local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/home/vscode/.local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/home/vscode/.local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/home/vscode/.local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


[0.94743 0.9494  0.94938 0.94848 0.94866]
mean: 0.94867  floor to beat: 0.86354


In [13]:
model.fit(X, y)
X_test = add_features(test.drop(columns=['id']))
preds = model.predict(X_test)

submission = pd.DataFrame({'id': test['id'], 'health_condition': preds})
submission.to_csv('../data/submission.csv', index=False)
print(submission['health_condition'].value_counts(normalize=True))

/home/vscode/.local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


health_condition
at-risk      0.883900
unhealthy    0.067053
fit          0.049048
Name: proportion, dtype: float64
